# 🛰️ GeoNexa — RemoteCLIP Foundation Model Fine-Tuning
### Contrastive Learning for Multi-Spectral Earth Observation Semantic Retrieval (SIH 2026)

This notebook fine-tunes **RemoteCLIP (ViT-B-32)** on multi-spectral satellite imagery across all Earth Observation domains on a GPU (T4 / A100 / V100 / L4).

---
### 🚀 End-to-End Workflow:
1. **Environment Setup**: Clone repository & install PyTorch / OpenCLIP dependencies
2. **Download Base RemoteCLIP Foundation Weights**: `RemoteCLIP-ViT-B-32.pt` (605 MB)
3. **Generate 4,100+ All-Domain Contrastive Training Pairs**: Urban, Agriculture, Mangroves, Rivers, Solar, Ports, Runways, Disasters
4. **GPU Contrastive Fine-Tuning**: 20-30 Epochs with InfoNCE loss + Cosine Annealing LR Schedule + AdamW on GPU (fp16)
5. **Quantitative Retrieval Evaluation**: Test Recall@1, Recall@5, Recall@10 across target queries
6. **Export & Download Checkpoint**: Auto-export deployment-ready `RemoteCLIP-ViT-B-32-finetuned.pt` and download to local PC

## 1. Environment & GPU Verification

In [ ]:
# Verify GPU
!nvidia-smi

# Clone or pull latest GeoNexa repository
!rm -rf GeoNexa
!git clone https://github.com/Amityush-lgtm/GeoNexa.git
%cd GeoNexa

# Install dependencies
!pip install -q open-clip-torch rasterio scipy pillow torchvision tqdm gdown faiss-cpu pydantic pydantic-settings fastapi uvicorn

## 2. Download Base Pretrained RemoteCLIP Weights

In [ ]:
!python scripts/download_models.py --force

## 3. Generate 4,100+ All-Domain Contrastive Training Pairs

In [ ]:
# Generate domain-adapted training pairs across all 11 EO categories
!python scripts/generate_all_queries_dataset.py

## 4. Run GPU Contrastive Fine-Tuning

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Starting fine-tuning on device: {device}')

# Run 20-epoch contrastive fine-tuning with mixed precision fp16
!python -m training.train \
    --train-data training_data/train_pairs.json \
    --val-data training_data/val_pairs.json \
    --checkpoint models/remoteclip-vit-b-32/RemoteCLIP-ViT-B-32.pt \
    --output-dir checkpoints \
    --epochs 20 \
    --batch-size 32 \
    --lr 2e-5 \
    --unfreeze-layers 4 \
    --device cuda \
    --num-workers 2 \
    --eval-every 2 \
    --save-every 2

## 5. Quantitative Retrieval Evaluation & Query Testing

In [ ]:
# Copy best checkpoint and evaluate retrieval precision
import shutil
shutil.copyfile('checkpoints/best_model.pt', 'models/RemoteCLIP-ViT-B-32-finetuned.pt')

# Rebuild index and test sample queries
!python scripts/rebuild_finetuned_index.py

## 6. Export Deployment Weights & Download to Local PC

In [ ]:
from google.colab import files

print('Exporting fine-tuned model checkpoint for local deployment...')
files.download('models/RemoteCLIP-ViT-B-32-finetuned.pt')